# 实战练习：使用 GRPO 微调模型

> **TIP**: 本练习由 LLM 微调专家 [@mlabonne](https://huggingface.co/mlabonne) 撰写。

现在进入实践环节！本练习将完整演示如何使用 GRPO 微调一个模型——从安装依赖到生成推理，完成一个端到端的训练流程。

**本练习使用的资源：**
- 模型：`SmolLM2-135M-Instruct`（135M 参数，适合有限硬件）
- 数据集：`mlabonne/smoltldr`（短篇故事集）
- 任务：训练模型生成接近 50 tokens 长度的摘要
- 硬件：单张 A10G GPU（约 1 小时完成训练，Google Colab 可用）

## 环境安装

In [ ]:
# 安装核心依赖
# datasets: HuggingFace 数据集库
# transformers: 模型加载和推理
# trl: TRL（Transformer Reinforcement Learning）训练库
# peft: LoRA 等参数高效微调方法
# accelerate: 多 GPU/混合精度训练支持
# bitsandbytes: 量化支持（int8/int4）
# wandb: 实验追踪和可视化
!pip install -qqq datasets==3.2.0 transformers==4.47.1 trl==0.14.0 peft==0.14.0 \
    accelerate==1.2.1 bitsandbytes==0.45.2 wandb==0.19.7 --progress-bar off

# flash-attn: Flash Attention 加速（显著提升训练效率，但安装较慢）
!pip install -qqq flash-attn --no-build-isolation --progress-bar off

In [ ]:
# 导入所有必要的库
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model       # LoRA 相关
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

## 实验追踪（Weights & Biases）

Weights & Biases（W&B）是强大的实验追踪工具，可以可视化训练曲线（reward、loss、KL 散度等）。

In [ ]:
import wandb

# 登录 W&B（需要 API key，可在 wandb.ai 注册获取）
# 也可以跳过 W&B，在 GRPOConfig 中设置 report_to=[] 即可
wandb.login()

# 或者用环境变量方式：
# import os
# os.environ["WANDB_API_KEY"] = "your-api-key"

## 加载数据集

In [ ]:
# 加载 smoltldr 数据集
# 该数据集包含短篇故事，适合用于摘要生成任务的 GRPO 训练
dataset = load_dataset("mlabonne/smoltldr")

# 查看数据集结构
print(dataset)
print()

# 查看第一个样本
print("第一个训练样本：")
print(dataset["train"][0])

## 加载模型

使用 `SmolLM2-135M-Instruct`，一个 135M 参数的小模型：
- 适合有限硬件（单卡 GPU 即可）
- 有 Instruct 版本（已经过指令微调）
- 如果有更强硬件，可尝试 `SmolLM2-1.7B`

In [ ]:
model_id = "HuggingFaceTB/SmolLM-135M-Instruct"

# 加载模型
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",              # 自动选择精度（通常是 bfloat16）
    device_map="auto",               # 自动分配到可用 GPU/CPU
    attn_implementation="flash_attention_2",  # 使用 Flash Attention 2 加速
)

# 加载对应的 tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

print(f"模型已加载：{model_id}")
print(f"模型参数量：{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 配置 LoRA

LoRA（Low-Rank Adaptation）通过在原始模型权重旁边添加低秩矩阵来实现参数高效微调：
- 只训练少量额外参数（通常 < 1% 的原始参数量）
- 大幅降低显存需求
- 训练速度更快

> **TIP**: 如果你不熟悉 LoRA，可以参考 [Chapter 11 第 3 节](https://huggingface.co/learn/course/en/chapter11/3)。

In [ ]:
# 配置 LoRA 参数
lora_config = LoraConfig(
    task_type="CAUSAL_LM",       # 因果语言模型任务
    r=16,                         # LoRA 的秩（rank），越大效果越好但参数越多
    lora_alpha=32,                # LoRA 缩放系数（通常设为 r 的 2 倍）
    target_modules="all-linear",  # 对所有线性层应用 LoRA（也可指定特定层）
)

# 将 LoRA 应用到模型
model = get_peft_model(model, lora_config)

# 打印可训练参数信息
model.print_trainable_parameters()
# 输出类似：trainable params: 2,686,976 || all params: 137,701,376 || trainable%: 1.95

## 定义奖励函数

本练习使用**长度控制奖励函数**：训练模型生成接近 50 tokens 的回答。

这是一个故意简化的奖励函数，用于演示 GRPO 的学习能力。实际应用中应使用更有意义的任务指标。

In [ ]:
# 目标生成长度（token 数量）
ideal_length = 50

def reward_len(completions, **kwargs):
    """
    长度控制奖励函数
    
    奖励值为负，越接近 ideal_length 奖励越高（越接近 0）
    例如：
      - 生成 50 tokens → 奖励 = 0（最高）
      - 生成 30 tokens → 奖励 = -20
      - 生成 100 tokens → 奖励 = -50
    """
    return [-abs(ideal_length - len(completion)) for completion in completions]


# 演示奖励函数行为
examples = [
    "short" * 2,            # 很短
    "medium " * 10,         # 接近目标长度
    "very long " * 20,      # 很长
]

print("奖励函数测试：")
for ex in examples:
    reward = reward_len([ex])[0]
    print(f"  长度={len(ex):4d} tokens → 奖励={reward:5.1f}")

## 定义训练参数

In [ ]:
# 配置 GRPO 训练参数
training_args = GRPOConfig(
    output_dir="GRPO",                    # 输出目录（检查点保存位置）
    
    # ---- 优化器参数 ----
    learning_rate=2e-5,                   # 学习率
    optim="adamw_8bit",                   # 使用 8-bit AdamW 优化器（节省显存）
    
    # ---- 批次参数 ----
    per_device_train_batch_size=8,        # 每卡 batch size（需能容纳 8 个生成结果）
    gradient_accumulation_steps=2,        # 梯度累积（等效 batch size = 16）
    
    # ---- 序列长度参数 ----
    max_prompt_length=512,                # prompt 最大长度（token 数）
    max_completion_length=96,             # 生成回答最大长度（token 数）
    
    # ---- GRPO 核心参数 ----
    num_generations=8,                    # 每个 prompt 生成 8 个候选答案
    
    # ---- 训练控制 ----
    num_train_epochs=1,                   # 训练 1 轮（约 1 小时）
    bf16=True,                            # 使用 bfloat16 精度
    
    # ---- 日志和评估 ----
    report_to=["wandb"],                  # 向 W&B 报告指标（也可设为 [] 禁用）
    logging_steps=1,                      # 每步都记录日志（便于观察训练过程）
    
    # ---- 其他 ----
    remove_unused_columns=False,          # 保留数据集中所有列（奖励函数可能需要）
)

print("训练参数配置完成")

## 启动训练

In [ ]:
# 初始化 GRPOTrainer
trainer = GRPOTrainer(
    model=model,                          # 已加载的模型（带 LoRA）
    reward_funcs=[reward_len],            # 奖励函数列表（可以传入多个）
    args=training_args,                   # 训练配置
    train_dataset=dataset["train"],       # 训练数据集
)

# 初始化 W&B 实验
wandb.init(project="GRPO")

# 开始训练！
# 在单张 A10G GPU 上约需 1 小时
trainer.train()

## 解读训练结果

训练过程中，`GRPOTrainer` 会记录以下关键指标：

### 奖励（Reward）

奖励值应随训练**逐渐向 0 靠拢**（因为我们的奖励函数最大值是 0）。
这是模型学习到生成接近目标长度的正面信号。

### 损失（Loss）

> **TIP**: GRPO 训练中，**loss 从 0 开始然后上升是完全正常的**，不代表训练出问题！

原因：GRPO 的 loss 正比于 KL 散度（当前策略与参考策略的差异）。随着模型优化以更好地匹配奖励函数，它会自然地偏离初始策略，导致 KL 散度增大，loss 升高。**loss 上升恰恰说明模型在学习**。

核心判断指标：
- ✅ `reward` 持续上升（向目标靠近）
- ✅ `reward_std` 非零（组内有多样性，梯度稳定）
- ⚠️ `kl` 过大（模型偏离参考策略太多，可增大 β）

## 发布模型到 HuggingFace Hub

In [ ]:
# 将 LoRA 权重合并到基础模型（得到完整的模型权重）
merged_model = trainer.model.merge_and_unload()

# 推送到 HuggingFace Hub
# private=False：公开模型（设为 True 则为私有）
# tags：添加标签，方便搜索和分类
merged_model.push_to_hub(
    "SmolGRPO-135M",
    private=False,
    tags=["GRPO", "Reasoning-Course"]
)

# 同时推送 tokenizer
tokenizer.push_to_hub("SmolGRPO-135M")

print("模型已成功发布到 HuggingFace Hub！")

## 使用模型进行推理

In [ ]:
# 准备测试文档（一篇关于猫的长文章）
prompt = """
# A long document about the Cat

The cat (Felis catus), also referred to as the domestic cat or house cat, is a small 
domesticated carnivorous mammal. It is the only domesticated species of the family Felidae.
Advances in archaeology and genetics have shown that the domestication of the cat occurred
in the Near East around 7500 BC. It is commonly kept as a pet and farm cat, but also ranges
freely as a feral cat avoiding human contact. It is valued by humans for companionship and
its ability to kill vermin. Its retractable claws are adapted to killing small prey species
such as mice and rats. It has a strong, flexible body, quick reflexes, and sharp teeth,
and its night vision and sense of smell are well developed. It is a social species,
but a solitary hunter and a crepuscular predator. Cat communication includes
vocalizations—including meowing, purring, trilling, hissing, growling, and grunting—as
well as body language. It can hear sounds too faint or too high in frequency for human ears,
such as those made by small mammals. It secretes and perceives pheromones.
"""

# 构造 chat 格式的消息
messages = [
    {"role": "user", "content": prompt},  # 用户输入长文档，期望模型生成摘要
]

print("准备推理输入完成")
print(f"文档长度：{len(prompt)} 字符")

In [ ]:
from transformers import pipeline

# 方式 1：从 Hub 加载已推送的模型
generator = pipeline("text-generation", model="SmolGRPO-135M")

# 方式 2：使用当前训练好的模型（不需要先推送）
# generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

# 生成参数配置
generate_kwargs = {
    "max_new_tokens": 256,    # 最多生成 256 个新 token
    "do_sample": True,        # 使用采样（而非贪心解码），增加多样性
    "temperature": 0.5,       # 温度控制多样性（越低越保守）
    "min_p": 0.1,             # 最小概率阈值（过滤低概率 token）
}

# 执行推理
generated_text = generator(messages, **generate_kwargs)

# 打印生成结果
print("=" * 50)
print("GRPO 微调后模型的摘要输出：")
print("=" * 50)
print(generated_text[0]["generated_text"])

## 本节小结

恭喜你完成了第一个 GRPO 微调练习！

### 你学到了什么

1. **完整的 GRPO 训练流程**：从安装依赖到发布模型
2. **LoRA 的使用**：通过参数高效微调减少显存需求
3. **奖励函数设计**：用长度控制函数演示了 GRPO 的学习过程
4. **训练结果解读**：理解了 loss 上升是正常现象，应关注 reward 指标

### 扩展挑战

- 尝试更大的模型（`SmolLM2-1.7B`）
- 设计更复杂的奖励函数（格式 + 内容质量）
- 在数学数据集上训练（如 GSM8K）
- 使用 `push_to_hub=True` 让训练过程中定期自动推送检查点

---

**下一节**：使用 Unsloth 加速 GRPO 训练，在免费 T4 GPU 上也能运行！